# 04 (Kaggle) - Decode & eval (RoPE)

Loads the RoPE checkpoint and reports **BLEU + chrF++ + TER** on the test set.

**Before running:** **GPU T4** + **Internet ON**. **Attach both** the **corpus dataset** (test files + tokenizer) and the **checkpoint dataset** (the autopushed `best.pt`), and set the two slugs below. `ModelConfig` uses `pos_encoding="rope"` to match the trained weights.

## 1. Repo + install

In [ ]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/kaggle/working/hindi-translator"

!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

In [ ]:
!pip install -e .

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Config + paths (fill in your dataset slugs)

In [ ]:
from nmt.config import DataConfig, ModelConfig, DecodeConfig

CORPUS_SLUG = "rishijain27/hindi-translator-dataset"   # corpus dataset (test files + tokenizer)
CKPT_SLUG   = "rishijain27/hindi-rope-ckpts"           # checkpoint dataset (autopushed best.pt)

DATA_DIR = f"/kaggle/input/{CORPUS_SLUG.split('/')[-1]}"
CKPT_PATH = f"/kaggle/input/{CKPT_SLUG.split('/')[-1]}/best.pt"

data_cfg = DataConfig(cache_dir=DATA_DIR)
model_cfg = ModelConfig(pos_encoding="rope")      # MUST match the trained model
decode_cfg = DecodeConfig(mode="beam", beam_size=5, length_penalty=0.6, batch_size=64, max_decode_len=128)
SPLIT = "test"                                     # "test" (~2507) or "dev" (~520, faster)
print("ckpt:", CKPT_PATH, "| mode:", decode_cfg.mode, "| split:", SPLIT)

## 3. Tokenizer + model

In [ ]:
from nmt.data.tokenizer import Tokenizer
from nmt.model.transformer import build_model

tok = Tokenizer.load(os.path.join(DATA_DIR, f"spm_{data_cfg.tokenizer_model}_{data_cfg.vocab_size}.model"))
model = build_model(model_cfg, tok).to(device)
print("tokenizer vocab:", tok.vocab_size, "| params:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 4. Load trained weights (EMA)

Load `best.pt`, then overlay the **EMA shadow** weights (what dev loss was measured on). Falls back to raw weights if no EMA.

In [ ]:
from nmt.train.ema import EMA

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])

if ckpt.get("ema"):
    ema = EMA(model)
    ema.load_state_dict(ckpt["ema"])
    ema.copy_to(model)
    print("applied EMA weights")
else:
    print("no EMA in checkpoint -> using raw weights")

model.eval()
print("checkpoint step:", ckpt["step"], "| best dev nll:", ckpt.get("best"))

## 5. Read the eval set

In [ ]:
from nmt.eval.evaluate import read_lines

src_path = os.path.join(DATA_DIR, f"{SPLIT}.clean.hi")
ref_path = os.path.join(DATA_DIR, f"{SPLIT}.clean.en")
srcs, refs = read_lines(src_path, ref_path)
print(f"{SPLIT}: {len(srcs)} sentence pairs")

## 6. Translate + score

Beam runs one sentence at a time (slower); try `SPLIT="dev"` first for a quick read.

In [ ]:
from nmt.eval.evaluate import evaluate
results = evaluate(model, tok, srcs, refs, decode_cfg)
results

## 7. Eyeball a few translations

In [ ]:
from nmt.decode.translate import translate

sample_src = srcs[:10]
sample_hyp = translate(model, sample_src, tok, decode_cfg)
for s, h, r in zip(sample_src, sample_hyp, refs[:10]):
    print("SRC:", s)
    print("HYP:", h)
    print("REF:", r)
    print("-" * 70)